# Κεφάλαιο 12: Εκπαίδευσε το μοντέλο σου

> «Δεν απέτυχα. Απλώς βρήκα 10.000 τρόπους που δεν λειτουργούν.»
> — **Τόμας Έντισον**, εφευρέτης

---

## Τι θα μάθεις

- Πώς μαθαίνουν τα μοντέλα να γράφουν προβλέποντας το επόμενο token
- Ποια είναι τα 5 βήματα για την εκπαίδευση κάθε νευρωνικού δικτύου
- Γιατί οι καμπύλες απώλειας δείχνουν αν το μοντέλο σου μαθαίνει
- Πώς να αποφεύγεις την υπερπροσαρμογή και να αναγνωρίζεις πότε να σταματήσεις την εκπαίδευση
- Πώς να αποθηκεύεις την κατάσταση εκπαίδευσης (checkpoint) για να μη χάνεις την πρόοδό σου
- Πόσο συναρπαστικό είναι να βλέπεις το μοντέλο σου να περνά από τις ασυναρτησίες σε κείμενο με συνοχή

---

## Προετοιμασία

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch12.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch12.ipynb).

Πρώτα, εγκατάστησε τα απαραίτητα πακέτα:

In [ ]:
# Εγκατάστησε τα απαραίτητα πακέτα στις εκδόσεις που δοκιμάστηκαν για το βιβλίο
# (βλ. Παράρτημα A). Στο επόμενο κελί θα εισαγάγεις απευθείας τα torch και transformers.
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== Εισαγωγή βιβλιοθηκών =====
import math
import urllib.request
from functools import partial
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

# Επίλεξε τη συσκευή για τους υπολογισμούς
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Συσκευή για τους υπολογισμούς: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ===== Αναπαραγωγιμότητα =====
import random
import numpy as np

def set_seed(seed=42):
    """Όρισε την ίδια αρχική τιμή για κάθε γεννήτρια τυχαίων αριθμών που χρησιμοποιούμε."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

## 1. Τα μέρη του μοντέλου από τα κεφάλαια 10-11

Εδώ ορίζουμε ξανά την αρχιτεκτονική του MiniGPT. Δεν φορτώνουμε προεκπαιδευμένα βάρη: εκπαιδεύουμε το μοντέλο από την αρχή στο TinyShakespeare, χωρίς το αρχείο JSONL του Κεφαλαίου 8.

In [ ]:
# ===== Προσοχή με πολλαπλές κεφαλές (από το Κεφάλαιο 10) =====
class MultiHeadAttention(nn.Module):
    """
    Αποδοτική υλοποίηση προσοχής με πολλαπλές κεφαλές: επεξεργάζεται όλες τις κεφαλές μαζί.
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "Το d_model πρέπει να διαιρείται ακριβώς με το num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Ενιαία γραμμική προβολή QKV (3 φορές πιο αποδοτική από τις χωριστές προβολές!)
        # Γιατί 3 * d_model; Επειδή υπολογίζουμε ταυτόχρονα τις προβολές Q, K και V
        # Με bias=True, όπως στο GPT-2, μπορούμε αργότερα να φορτώσουμε τους προεκπαιδευμένους όρους μετατόπισής του
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Γραμμική προβολή εξόδου
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout στα βάρη προσοχής
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Βήμα 1: Υπολόγισε τις προβολές Q, K και V με μία πράξη =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Βήμα 2: Ξεχώρισε τα Q, K και V και προσάρμοσε το σχήμα για τις πολλαπλές κεφαλές =====
        # Άλλαξε το σχήμα σε (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Άλλαξε τη σειρά των αξόνων σε (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Ξεχώρισε τους τανυστές Q, K και V: καθένας έχει σχήμα (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Βήμα 3: Υπολόγισε την προσοχή με κλιμακωτό εσωτερικό γινόμενο για όλες τις κεφαλές μαζί =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Εφάρμοσε την αιτιακή μάσκα, αν έχει δοθεί
        if mask is not None:
            # Πρόσθεσε άξονες για να ισχύει η μάσκα για κάθε δείγμα και κάθε κεφαλή: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Σταθμισμένο άθροισμα των τιμών
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Βήμα 4: Συνένωσε τις εξόδους των κεφαλών =====
        # Αντάλλαξε τους άξονες ώστε το σχήμα να γίνει (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Άλλαξε το σχήμα σε (batch, seq, d_model) για να συνενώσεις τις εξόδους των κεφαλών
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Βήμα 5: Τελική γραμμική προβολή =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Δοκίμασέ το

print("Ορίστηκε η κλάση MultiHeadAttention!")

In [ ]:
# ===== Δίκτυο πρόσθιας τροφοδότησης (από το Κεφάλαιο 10) =====
class FeedForward(nn.Module):
    """
    Δίκτυο πρόσθιας τροφοδότησης ανά θέση.
    Επεξεργάζεται κάθε θέση ανεξάρτητα (με τα ίδια βάρη για όλες τις θέσεις).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768 για το GPT-2 small)
            d_ff: Διάσταση του κρυφού επιπέδου (συνήθως 4 * d_model = 3072)
            dropout: Πιθανότητα μηδενισμού μιας ενεργοποίησης κατά την εκπαίδευση
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), αυξάνει τη διάσταση του διανύσματος
        x = F.gelu(x, approximate="tanh")  # Μη γραμμική ενεργοποίηση (η παραλλαγή της GELU που χρησιμοποιεί το GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), προβάλλει το διάνυσμα πίσω στην αρχική διάσταση
        return x


# Δοκίμασέ το

print("Ορίστηκε η κλάση FeedForward!")

In [ ]:
# ===== Μπλοκ Transformer (από το Κεφάλαιο 10) =====
class TransformerBlock(nn.Module):
    """
    Πλήρες μπλοκ Transformer με προσοχή με πολλαπλές κεφαλές, δίκτυο πρόσθιας τροφοδότησης,
    υπολειμματικές συνδέσεις και κανονικοποίηση επιπέδου πριν από κάθε υποεπίπεδο, όπως στο GPT-2.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768)
            num_heads: Αριθμός κεφαλών προσοχής (12)
            d_ff: Εσωτερική διάσταση του δικτύου πρόσθιας τροφοδότησης (3072 = 4 * d_model)
            dropout: Πιθανότητα απενεργοποίησης
        """
        super().__init__()

        # Κανονικοποίηση επιπέδου (πριν από κάθε υποεπίπεδο)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Προσοχή με πολλαπλές κεφαλές
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Δίκτυο πρόσθιας τροφοδότησης
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Τυχαία απενεργοποίηση (dropout) μετά από κάθε υποεπίπεδο
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Ορίσματα:
            x: Διανυσματικές αναπαραστάσεις εισόδου (batch, seq, d_model)
            mask: Αιτιακή μάσκα (seq, seq) ή (1, 1, seq, seq)

        Επιστρέφει:
            x: Διανυσματικές αναπαραστάσεις εξόδου (batch, seq, d_model)
            attn_weights: Βάρη προσοχής (batch, heads, seq, seq)
        """
        # ===== Προσοχή με πολλαπλές κεφαλές και υπολειμματική σύνδεση =====
        # Κανονικοποίησε την είσοδο πριν υπολογίσεις την προσοχή
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Υπολειμματική σύνδεση

        # ===== Δίκτυο πρόσθιας τροφοδότησης με υπολειμματική σύνδεση =====
        # Κανονικοποίησε την είσοδο πριν την περάσεις στο FFN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Υπολειμματική σύνδεση

        return x, attn_weights


# Δοκίμασε ένα πλήρες μπλοκ Transformer

print("Ορίστηκε η κλάση TransformerBlock!")

In [ ]:
# ===== Ρυθμίσεις GPT (από το Κεφάλαιο 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Ρυθμίσεις του μοντέλου MiniGPT."""
    vocab_size: int = 50257      # Μέγεθος λεξιλογίου του GPT-2
    max_seq_len: int = 1024      # Μέγιστο μήκος συμφραζομένων
    embed_dim: int = 768         # Διάσταση της διανυσματικής αναπαράστασης (d_model)
    num_heads: int = 12          # Αριθμός κεφαλών προσοχής
    num_layers: int = 12         # Αριθμός μπλοκ Transformer
    d_ff: int = 3072             # Πλάτος του κρυφού επιπέδου του δικτύου πρόσθιας τροφοδότησης (4 * embed_dim)
    dropout: float = 0.1         # Πιθανότητα απενεργοποίησης (dropout)

    def __post_init__(self):
        """Ελέγχει τις ρυθμίσεις μετά την αρχικοποίηση."""
        assert self.embed_dim % self.num_heads == 0, \
            f"Η τιμή του embed_dim ({self.embed_dim}) πρέπει να διαιρείται ακριβώς με " \
            f"την τιμή του num_heads ({self.num_heads})"

In [ ]:
# ===== Μοντέλο MiniGPT (από το Κεφάλαιο 11) =====
class MiniGPT(nn.Module):
    """
    Ένα απλό γλωσσικό μοντέλο τύπου GPT.
    Συνδυάζει διανυσματικές αναπαραστάσεις (embeddings), μπλοκ Transformer και μια κεφαλή πρόβλεψης του επόμενου token.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Διανυσματικές αναπαραστάσεις token και θέσεων (Κεφάλαιο 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Μπλοκ Transformer (Κεφάλαιο 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Τελική κανονικοποίηση επιπέδου =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Κεφαλή πρόβλεψης του επόμενου token =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Κοινά βάρη =====
        # Χρησιμοποίησε τα ίδια βάρη για τις αναπαραστάσεις token και την προβολή εξόδου
        self.lm_head.weight = self.token_embed.weight

        # Αρχικοποίησε τα βάρη
        self._init_weights()

    def _init_weights(self):
        """Αρχικοποιεί τα βάρη με μικρές τυχαίες τιμές."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Περνά την είσοδο από τα επίπεδα του μοντέλου για να υπολογίσει την έξοδο.

        Ορίσματα:
            token_ids: ID των token εισόδου (batch, seq)
            return_attention: Αν θα επιστραφούν και τα βάρη προσοχής

        Επιστρέφει:
            logits: Βαθμοί πριν από τη softmax για κάθε token του λεξιλογίου (batch, seq, vocab_size)
            attention_weights: Προαιρετική λίστα με τα βάρη προσοχής κάθε μπλοκ
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Διανυσματικές αναπαραστάσεις =====
        # Αναπαραστάσεις των token
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Αναπαραστάσεις των θέσεων
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Πρόσθεσε τις αναπαραστάσεις και εφάρμοσε dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Δημιουργία αιτιακής μάσκας =====
        # Κάτω τριγωνική μάσκα: κάθε θέση βλέπει μόνο τον εαυτό της και τις προηγούμενες θέσεις
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Διέλευση από τα μπλοκ Transformer =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Τελική κανονικοποίηση επιπέδου =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Κεφαλή πρόβλεψης του επόμενου token =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Συνεχίζει την ακολουθία input_ids με νέα token (η έξοδος περιλαμβάνει και την προτροπή).

        Ορίσματα:
            input_ids: ID των token της προτροπής (prompt), με σχήμα (batch, seq)
            max_new_tokens: Μέγιστος αριθμός νέων token
            temperature: Θερμοκρασία δειγματοληψίας (χρησιμοποιείται μόνο όταν do_sample=True)
            do_sample: Αν είναι True, επιλέγει token με δειγματοληψία από την κατανομή· αν είναι False, επιλέγει το token με τον μεγαλύτερο βαθμό
            eos_token_id: Σταματά νωρίτερα αν όλες οι ακολουθίες παραγάγουν αυτό το token στο ίδιο βήμα

        Επιστρέφει:
            ID των token με σχήμα (batch, seq + πλήθος νέων token)
        """
        self.eval()

        # Οι αναπαραστάσεις θέσεων καλύπτουν μόνο max_seq_len θέσεις
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Το μήκος της προτροπής ({input_ids.shape[1]} token) μαζί με τα νέα token "
                f"που ζήτησες (max_new_tokens={max_new_tokens}) ξεπερνά το όριο των {limit} token "
                "του μοντέλου. Συντόμευσε την προτροπή ή μείωσε το max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids

## 2. Κατέβασε το σύνολο δεδομένων

Θα χρησιμοποιήσουμε το TinyShakespeare, ένα σύνολο δεδομένων αρκετά μικρό ώστε να εκπαιδεύσεις το μοντέλο σε λίγα λεπτά. Το σώμα κειμένων και οι προτροπές (prompts) παραμένουν στα αγγλικά, ώστε ο tokenizer να δίνει τις ίδιες εισόδους για την εκπαίδευση και τη σύγκριση των αποτελεσμάτων.

In [ ]:
# Κατέβασε το σύνολο δεδομένων TinyShakespeare
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Μέγεθος συνόλου δεδομένων: {len(text):,} χαρακτήρες")
print(f"\nΑπόσπασμα:\n{text[:500]}")

## 3. Δημιούργησε το σύνολο δεδομένων και το DataLoader

Πρόσεξε τι *δεν* κάνει η `create_lm_labels`: δεν εφαρμόζει καμία μάσκα. Στους βρόχους εκπαίδευσης και αξιολόγησης χρησιμοποιούμε τη μάσκα προσοχής για να εξαιρέσουμε από τον υπολογισμό της απώλειας μόνο τις θέσεις συμπλήρωσης (`targets[attention_mask[:, 1:] == 0] = -100`). Έτσι, όταν ένα token EOS δηλώνει πραγματικό τέλος κειμένου, παραμένει στόχος πρόβλεψης. Το GPT-2 χρησιμοποιεί το ίδιο token EOS (ID 50256) και για τη συμπλήρωση. Μια μάσκα που ελέγχει μόνο την τιμή του ID δεν μπορεί να ξεχωρίσει τις δύο περιπτώσεις: θα εξαιρούσε χωρίς προειδοποίηση κάθε πραγματικό τέλος κειμένου από τους στόχους πρόβλεψης και έτσι θα μάθαινε στο μοντέλο να μη σταματά ποτέ.

In [ ]:
def create_lm_labels(token_ids):
    """
    Δημιούργησε εισόδους και στόχους για την πρόβλεψη του επόμενου token.

    Ορίσματα:
        token_ids: ID των token με σχήμα (batch, seq)

    Επιστρέφει:
        inputs: (batch, seq-1), όλα τα token εκτός από το τελευταίο
        targets: (batch, seq-1), όλα τα token εκτός από το πρώτο
    """
    inputs = token_ids[:, :-1]    # Όλα τα token εκτός από το τελευταίο
    targets = token_ids[:, 1:].clone()  # .clone(): οι βρόχοι γράφουν -100 στις θέσεις συμπλήρωσης του αντιγράφου

    # Δεν εφαρμόζουμε μάσκα εδώ: οι βρόχοι χρησιμοποιούν τη μάσκα προσοχής για να αγνοήσουν τις θέσεις συμπλήρωσης.
    # Έτσι, ένα πραγματικό token τέλους (EOS) παραμένει στόχος, παρόλο που το GPT-2 χρησιμοποιεί το ίδιο ID (50256) για συμπλήρωση.
    return inputs, targets


from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """Απλό σύνολο δεδομένων που επιστρέφει τμήματα κειμένου."""

    def __init__(self, text, chunk_size=512):
        """
        Ορίσματα:
            text: Ολόκληρο το κείμενο ως συμβολοσειρά
            chunk_size: Πλήθος χαρακτήρων ανά τμήμα
        """
        self.chunks = []

        # Κράτησε και το τελευταίο πλήρες τμήμα. Αγνόησε μόνο το υπόλοιπο αν είναι μικρότερο.
        for i in range(0, len(text) - chunk_size + 1, chunk_size):
            self.chunks.append(text[i:i + chunk_size])

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


assert list(TextDataset("abcdefgh", chunk_size=4)) == ["abcd", "efgh"]
assert list(TextDataset("abcdefg", chunk_size=4)) == ["abcd"]


# Δημιούργησε το σύνολο δεδομένων


def collate_fn(batch, tokenizer, max_length=128):
    """
    Κωδικοποίησε τις συμβολοσειρές μιας παρτίδας σε token και συμπλήρωσε τις ακολουθίες ώστε να έχουν το ίδιο μήκος.

    Ορίσματα:
        batch: Λίστα με συμβολοσειρές κειμένου
        tokenizer: Ο tokenizer που θα χρησιμοποιήσεις
        max_length: Μέγιστο μήκος ακολουθίας

    Επιστρέφει:
        input_ids: ID των token μετά τη συμπλήρωση (batch, max_length)
        attention_mask: 1 για τα πραγματικά token, 0 για τις θέσεις συμπλήρωσης
    """
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]


# Φόρτωσε τον tokenizer (το GPT-2 δεν έχει από προεπιλογή token συμπλήρωσης· χρησιμοποίησε το token τέλους κειμένου)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Δημιούργησε το σύνολο δεδομένων: στο TinyShakespeare, κάθε τμήμα 512 χαρακτήρων αντιστοιχεί κατά μέσο όρο σε ~156
# token του GPT-2, οπότε το όριο max_length=128 κόβει το τέλος των περισσότερων τμημάτων.
# Αυτό δεν πειράζει στην προεκπαίδευση: κάθε τμήμα εξακολουθεί να δίνει πολλούς στόχους πρόβλεψης.
dataset = TextDataset(text, chunk_size=512)
print(f"Πλήθος τμημάτων: {len(dataset)}")

In [ ]:
# Χώρισε τα δεδομένα σε σύνολα εκπαίδευσης και επικύρωσης
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Εκπαίδευση: {len(train_dataset)}, Επικύρωση: {len(val_dataset)}")

# Δημιούργησε ένα DataLoader για κάθε σύνολο
collate = partial(collate_fn, tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# Έλεγξε μία παρτίδα
input_ids, attention_mask = next(iter(train_loader))
print(f"Σχήμα του τανυστή input_ids στην παρτίδα: {input_ids.shape}")

## 4. Φτιάξε το μοντέλο

In [ ]:
# Ρύθμισε ένα μικρό μοντέλο για γρήγορη εκπαίδευση
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

model = MiniGPT(config).to(device)
print(f"Παράμετροι: {sum(p.numel() for p in model.parameters()):,}")

## 5. Το «πριν»: το μοντέλο χωρίς εκπαίδευση

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    """
    Παράγει κείμενο με ρυθμιζόμενη θερμοκρασία δειγματοληψίας.

    Θερμοκρασία:
    - < 1.0: Ευνοούνται περισσότερο τα πιο πιθανά token
    - = 1.0: Η κατανομή πιθανοτήτων παραμένει αμετάβλητη
    - > 1.0: Η κατανομή γίνεται πιο επίπεδη και οι επιλογές πιο τυχαίες

    Ορίσματα:
        model: Το εκπαιδευμένο μοντέλο
        tokenizer: Το εργαλείο που τεμαχίζει και κωδικοποιεί το κείμενο
        prompt: Η προτροπή (prompt), δηλαδή το αρχικό κείμενο
        max_new_tokens: Ο μέγιστος αριθμός νέων token που θα παραχθούν
        temperature: Η θερμοκρασία δειγματοληψίας

    Επιστρέφει:
        Το κείμενο που προέκυψε, ως συμβολοσειρά
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=True: η μέθοδος διαιρεί τα logits με τη θερμοκρασία και μετά επιλέγει το επόμενο token με δειγματοληψία
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(output_ids[0])


# Δες τι κείμενο παράγει το μοντέλο πριν εκπαιδευτεί
print("Πριν από την εκπαίδευση (τυχαία βάρη):")
print("="*50)
prompt = "The king"
print(f"Προτροπή: «{prompt}»")
print(f"Έξοδος: {generate(model, tokenizer, prompt)}")
print("\n(Το κείμενο δεν βγάζει νόημα — το μοντέλο δεν έχει μάθει τίποτα ακόμα!)")

## 6. Προετοιμασία για την εκπαίδευση

In [ ]:
# Υπερπαράμετροι εκπαίδευσης
num_epochs = 3
learning_rate = 3e-4
warmup_steps = 100

total_steps = len(train_loader) * num_epochs
print(f"Συνολικά βήματα εκπαίδευσης: {total_steps}")

# Βελτιστοποιητής και ρύθμιση του ρυθμού μάθησης
# Η βοηθητική συνάρτηση από το Κεφάλαιο 11 χωρίζει τις παραμέτρους σε ομάδες
def get_parameter_groups(model, weight_decay=0.1):
    decay_params = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Με τη βοηθητική συνάρτηση από το Κεφάλαιο 11 εφαρμόζουμε εξασθένηση βαρών στους πίνακες,
# αλλά όχι στους όρους μετατόπισης ή στα διανύσματα του LayerNorm
optimizer = AdamW(get_parameter_groups(model, weight_decay=0.01),
                  lr=learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

## 7. Συναρτήσεις εκπαίδευσης και αξιολόγησης

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, clip_norm=1.0):
    """
    Εκπαιδεύει το μοντέλο για μία εποχή.

    Επιστρέφει:
        Τη μέση απώλεια των παρτίδων της εποχής
    """
    model.train()  # Ενεργοποίησε το dropout
    total_loss = 0

    progress = tqdm(dataloader, desc="Εκπαίδευση")
    for input_ids, attention_mask in progress:
        # Μετέφερε τα δεδομένα στη συσκευή
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Δημιούργησε τις εισόδους και τους στόχους: κάθε στόχος είναι το επόμενο token
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # Σήμανε τις θέσεις συμπλήρωσης με -100 ώστε να μη μετρήσουν στην απώλεια

        # ===== Τα 5 βήματα της εκπαίδευσης =====

        # Βήμα 1: Καθάρισε τις κλίσεις από το προηγούμενο βήμα
        optimizer.zero_grad(set_to_none=True)

        # Βήμα 2: Πέρασε τις εισόδους από το μοντέλο
        logits = model(inputs)

        # Βήμα 3: Υπολόγισε την απώλεια
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # Η reshape λειτουργεί ακόμη και για τομές που δεν είναι συνεχόμενες στη μνήμη
            ignore_index=-100  # Αγνόησε τις θέσεις συμπλήρωσης
        )

        # Βήμα 4: Υπολόγισε τις κλίσεις με οπισθοδιάδοση
        loss.backward()

        # Περιορισμός κλίσεων: μετά την backward() και πριν από τη step(), περιόρισε τη συνολική νόρμα των κλίσεων.
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        # Βήμα 5: Ενημέρωσε τα βάρη
        optimizer.step()
        scheduler.step()

        # ===== Τέλος των 5 βημάτων =====

        total_loss += loss.item()

        # Ενημέρωσε τη γραμμή προόδου
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """
    Αξιολογεί το μοντέλο σε ένα σύνολο δεδομένων.

    Επιστρέφει:
        avg_loss: Μέση απώλεια διασταυρούμενης εντροπίας ανά token
        perplexity: Εκθετική συνάρτηση της μέσης απώλειας, για ευκολότερη ερμηνεία
    """
    model.eval()  # Απενεργοποίησε το dropout
    total_loss = 0
    total_tokens = 0

    for input_ids, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # Αγνόησε τις θέσεις συμπλήρωσης στον υπολογισμό της απώλειας

        logits = model(inputs)

        # Υπολόγισε την απώλεια
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # Η reshape λειτουργεί ακόμη και για τομές που δεν είναι συνεχόμενες στη μνήμη
            ignore_index=-100,
            reduction='sum'  # Υπολόγισε το άθροισμα αντί για τον μέσο όρο
        )

        # Μέτρησε μόνο τα token που δεν είναι συμπλήρωση, ώστε να υπολογιστεί σωστά η perplexity
        total_loss += loss.item()
        total_tokens += (targets != -100).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


def should_stop_early(val_losses, patience=3):
    """
    Ελέγχει αν πρέπει να σταματήσει νωρίτερα η εκπαίδευση.

    Ορίσματα:
        val_losses: Λίστα με τις απώλειες επικύρωσης ανά εποχή
        patience: Αριθμός διαδοχικών αυξήσεων της απώλειας που οδηγούν σε διακοπή

    Επιστρέφει:
        True αν πρέπει να σταματήσει η εκπαίδευση
    """
    if len(val_losses) < patience + 1:
        return False

    # Έλεγξε αν η απώλεια αυξήθηκε σε καθεμία από τις τελευταίες 'patience' εποχές
    recent = val_losses[-(patience + 1):]
    for i in range(1, len(recent)):
        if recent[i] <= recent[i-1]:
            return False

    return True

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, path):
    """
    Αποθηκεύει την κατάσταση εκπαίδευσης (checkpoint) μαζί με τις καταστάσεις των γεννητριών τυχαίων αριθμών, ώστε η εκπαίδευση να συνεχιστεί ακριβώς από εκεί που σταμάτησε.

    Ορίσματα:
        model: Το μοντέλο
        optimizer: Ο βελτιστοποιητής (διατηρεί την κατάσταση της ορμής)
        scheduler: Ο μηχανισμός που ρυθμίζει τον ρυθμό μάθησης
        epoch: Ο αριθμός της τρέχουσας εποχής
        train_loss: Η απώλεια εκπαίδευσης για αυτή την εποχή
        val_loss: Η απώλεια επικύρωσης για αυτή την εποχή
        path: Η διαδρομή όπου θα αποθηκευτεί το αρχείο
    """
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        # Το dropout και το ανακάτεμα των δεδομένων πρέπει να συνεχίσουν την ίδια ακολουθία τυχαίων αριθμών.
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
    }, path)
    print(f"Η κατάσταση εκπαίδευσης αποθηκεύτηκε: {path}")


def load_checkpoint(model, optimizer, scheduler, path, device):
    """
    Φορτώνει την αποθηκευμένη κατάσταση εκπαίδευσης και επαναφέρει τις καταστάσεις των γεννητριών τυχαίων αριθμών.

    Επιστρέφει:
        epoch: Η εποχή από την οποία θα συνεχιστεί η εκπαίδευση
        train_loss: Η τελευταία απώλεια εκπαίδευσης
        val_loss: Η τελευταία απώλεια επικύρωσης
    """
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Οι τανυστές με τις καταστάσεις των γεννητριών πρέπει να βρίσκονται στη CPU: το map_location=device μπορεί να τους μεταφέρει σε άλλη συσκευή.
    random.setstate(checkpoint['rng_python'])
    np.random.set_state(checkpoint['rng_numpy'])
    torch.set_rng_state(checkpoint['rng_torch'].cpu())
    if checkpoint['rng_cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint['rng_cuda']])
    if checkpoint['rng_mps'] is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(checkpoint['rng_mps'].cpu())

    print(f"Φορτώθηκε η κατάσταση εκπαίδευσης της εποχής {checkpoint['epoch']}")

    return checkpoint['epoch'], checkpoint['train_loss'], checkpoint['val_loss']

Εδώ υπάρχει ένας συμβιβασμός: με `weights_only=True` δεν επιτρέπεται η εκτέλεση αυθαίρετου κώδικα pickle κατά τη φόρτωση. Προτίμησε αυτή τη ρύθμιση όταν φορτώνεις μια αποθηκευμένη κατάσταση εκπαίδευσης (checkpoint) από πηγή που δεν εμπιστεύεσαι. Όμως η κατάσταση της γεννήτριας τυχαίων αριθμών της NumPy περιέχει βαθμωτές τιμές της NumPy, τις οποίες ο ασφαλής μηχανισμός φόρτωσης απορρίπτει. Γι’ αυτό η έκδοση που συνεχίζει την εκπαίδευση ακριβώς από εκεί που σταμάτησε χρησιμοποιεί `weights_only=False`. Φόρτωσε με αυτόν τον τρόπο μόνο αρχεία που δημιούργησες εσύ ή που εμπιστεύεσαι ρητά. Για όσα κατέβασες, προτίμησε το `weights_only=True`, ακόμη κι αν έτσι δεν επανέλθουν οι γεννήτριες τυχαίων αριθμών στην αποθηκευμένη τους κατάσταση. Για να συνεχιστεί η εκπαίδευση με αποτελέσματα ίδια μέχρι και το τελευταίο bit, χρειάζονται δύο ακόμη πράγματα: ρύθμισε ξανά τον προγραμματιστή ρυθμού μάθησης για τον ίδιο συνολικό αριθμό βημάτων όπως στην αρχική εκτέλεση και επεξεργάσου τις παρτίδες με την ίδια σειρά.

## 8. Βρόχος εκπαίδευσης

In [ ]:
# Ώρα για εκπαίδευση!
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Εποχή {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")

    # Εκπαίδευσε το μοντέλο
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Αξιολόγησε το μοντέλο στο σύνολο επικύρωσης
    val_loss, perplexity = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    print(f"\nΑπώλεια εκπαίδευσης: {train_loss:.4f}")
    print(f"Απώλεια επικύρωσης:  {val_loss:.4f}")
    print(f"Περιπλοκότητα: {perplexity:.1f}")

    # Αποθήκευσε την κατάσταση εκπαίδευσης (checkpoint) αν μειώθηκε η ελάχιστη απώλεια επικύρωσης
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch,
                       train_loss, val_loss, "best_model.pt")

print("\nΗ εκπαίδευση ολοκληρώθηκε!")
print(f"Χαμηλότερη απώλεια επικύρωσης: {best_val_loss:.4f}")

## 9. Σχεδίασε τις καμπύλες απώλειας

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-o', label='Απώλεια εκπαίδευσης')
plt.plot(epochs, val_losses, 'r-o', label='Απώλεια επικύρωσης')
plt.xlabel('Εποχή')
plt.ylabel('Απώλεια')
plt.title('Απώλεια εκπαίδευσης και επικύρωσης')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Η στιγμή που περίμενες: το μοντέλο σου παράγει κείμενο!

In [ ]:
# Φόρτωσε το καλύτερο μοντέλο. Αποθήκευσες εσύ την κατάσταση εκπαίδευσης (checkpoint), οπότε το weights_only=False είναι ασφαλές εδώ.
# Για να διαβάσεις και τις καταστάσεις των γεννητριών τυχαίων αριθμών, χρειάζεσαι πλήρη φόρτωση· δες την ενότητα για τις αποθηκευμένες καταστάσεις εκπαίδευσης.
checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Φορτώθηκε η αποθηκευμένη κατάσταση εκπαίδευσης από την εποχή {checkpoint['epoch'] + 1}")
print(f"Απώλεια επικύρωσης: {checkpoint['val_loss']:.4f}")

In [ ]:
print("\n" + "="*60)
print("Μετά την εκπαίδευση σε κείμενα του Σαίξπηρ:")
print("="*60)

prompts = [
    "The king",
    "To be or not to be",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]

for prompt in prompts:
    output = generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8)
    print(f"\nΠροτροπή: «{prompt}»")
    print(f"Παραγόμενο κείμενο: {output}")
    print("-"*40)

## 11. Σύγκρινε το πριν με το μετά

In [ ]:
# Δημιούργησε ένα νέο, μη εκπαιδευμένο μοντέλο για σύγκριση
set_seed(123)  # Άλλαξε τον σπόρο της γεννήτριας για να πάρεις διαφορετικά τυχαία αρχικά βάρη
untrained_model = MiniGPT(config).to(device)
untrained_model.eval()

prompt = "The fair maiden"

print("="*60)
print(f"Προτροπή: «{prompt}»")
print("="*60)
print("\nΠΡΙΝ (χωρίς εκπαίδευση):")
print(generate(untrained_model, tokenizer, prompt, max_new_tokens=30))
print("\nΜΕΤΑ (με εκπαίδευση σε κείμενα του Σαίξπηρ):")
print(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8))
print("\nΊδια αρχιτεκτονική. Ίδιος κώδικας. Η εκπαίδευση κάνει τη διαφορά!")

## 12. Σύγκριση διαφορετικών τιμών θερμοκρασίας δειγματοληψίας

In [ ]:
prompt = "The noble lord"

print(f"Προτροπή: «{prompt}»\n")
for temp in [0.5, 0.8, 1.0, 1.5]:
    output = generate(model, tokenizer, prompt, max_new_tokens=30, temperature=temp)
    print(f"Θερμοκρασία δειγματοληψίας {temp}:")
    print(f"  {output}")
    print()

## Σύνοψη

**Τι φτιάξαμε:**

1. **Μετατόπιση των ετικετών** για την πρόβλεψη του επόμενου token
2. **DataLoaders** που κωδικοποιούν αποδοτικά το κείμενο σε token και το οργανώνουν σε παρτίδες
3. **Βρόχο εκπαίδευσης σε 5 βήματα**: zero_grad → forward → loss → backward → step
4. **Αξιολόγηση** με την απώλεια επικύρωσης και την περιπλοκότητα (perplexity)
5. **Αποθήκευση της κατάστασης εκπαίδευσης** (checkpoint) για να μπορείς να συνεχίσεις από εκεί που σταμάτησες
6. **Παραγωγή κειμένου** με ρύθμιση της θερμοκρασίας δειγματοληψίας

**Βασικές έννοιες:**

- Η εκπαίδευση είναι ένας βρόχος ελέγχου με ανάδραση: μετράς το σφάλμα, προσαρμόζεις τα βάρη και επαναλαμβάνεις
- Η απώλεια διασταυρούμενης εντροπίας τιμωρεί περισσότερο μια λάθος πρόβλεψη όταν το μοντέλο είναι σίγουρο γι’ αυτήν παρά όταν είναι αβέβαιο
- Υπερπροσαρμογή = το μοντέλο απομνημονεύει τα δεδομένα εκπαίδευσης (απώλεια εκπαίδευσης ↓, απώλεια επικύρωσης ↑)
- Η περιπλοκότητα μετρά πόσο «εκπλήσσεται» το μοντέλο από τα token που βλέπει· όσο χαμηλότερη είναι, τόσο το καλύτερο
- Η θερμοκρασία δειγματοληψίας ρυθμίζει την ποικιλία του παραγόμενου κειμένου

**Στη συνέχεια:** Κράτησε το δικό σου `best_model.pt`. Για να προσαρμόσεις το ίδιο μοντέλο στο Κεφάλαιο 13, τοποθέτησε το αρχείο στον κατάλογο εργασίας του notebook και ακολούθησε τον Τρόπο Α.

## Ασκήσεις

Τα τρία πειράματα του notebook έχουν δική τους αρίθμηση. Για να δουλέψεις με το σώμα κειμένων για τον κήπο του Κεφαλαίου 12 και να χωρίσεις τα έγγραφα σε σύνολα, ακολούθησε την **Άσκηση 1 του βιβλίου**. Για να συνεχίσεις την εκπαίδευση από την αποθηκευμένη κατάσταση εκπαίδευσης (checkpoint) μετά από επανεκκίνηση, ακολούθησε την **Άσκηση 4 του βιβλίου**. Το παρακάτω πείραμα με τον ρυθμό μάθησης αντιστοιχεί στην **Άσκηση 2 του βιβλίου**.

### Άσκηση 1: Πείραμα με τον ρυθμό μάθησης

Δοκίμασε διαφορετικές τιμές για τον ρυθμό μάθησης και σύγκρινε τα αποτελέσματα.

In [ ]:
# Γράψε εδώ τον κώδικά σου
# Εκπαίδευσε το μοντέλο τρεις φορές: με lr=1e-5, με lr=3e-4 και με lr=1e-2
# Σύγκρινε τις καμπύλες απώλειας
# Ποιος ρυθμός μάθησης δίνει το καλύτερο αποτέλεσμα;

### Άσκηση 2: Περισσότερες εποχές

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ
# Εκπαίδευσε το μοντέλο για 5-10 εποχές αντί για 3
# Συνεχίζει να βελτιώνεται το μοντέλο;
# Βλέπεις σημάδια υπερπροσαρμογής;

### Άσκηση 3: Σύγκρινε μοντέλα διαφορετικού μεγέθους

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ
# Φτιάξε ένα μικρότερο μοντέλο (2 επίπεδα, διανύσματα 128 διαστάσεων)
# Φτιάξε ένα μεγαλύτερο μοντέλο (6 επίπεδα, διανύσματα 384 διαστάσεων)
# Σύγκρινε τα δύο μοντέλα ως προς την ταχύτητα εκπαίδευσης και την τελική περιπλοκότητα (perplexity)